In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")

db = client.sample_mflix
movies = db.movies

print(movies)

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'sample_mflix'), 'movies')


In [2]:
# find() => aggregation Framework 문법을 사용하면 pipeline 생성해서 복잡한 데이터처리를 순차적으로 사용할 수 있다.

# 1. $match => 특정조건에 해당되는 문서를 찾아오고자 할 때
# 2. $limit => find().limit(2) 동일한 기능

for movie in movies.aggregate([{"$match": {"genres": "Action"}}, {"$limit": 1}]) :
    print(movie)

{'_id': ObjectId('573a1390f29313caabcd5293'), 'plot': "Young Pauline is left a lot of money when her wealthy uncle dies. However, her uncle's secretary has been named as her guardian until she marries, at which time she will officially take ...", 'genres': ['Action'], 'runtime': 199, 'cast': ['Pearl White', 'Crane Wilbur', 'Paul Panzer', 'Edward Josè'], 'num_mflix_comments': 1, 'poster': 'https://m.media-amazon.com/images/M/MV5BMzgxODk1Mzk2Ml5BMl5BanBnXkFtZTgwMDg0NzkwMjE@._V1_SY1000_SX677_AL_.jpg', 'title': 'The Perils of Pauline', 'fullplot': 'Young Pauline is left a lot of money when her wealthy uncle dies. However, her uncle\'s secretary has been named as her guardian until she marries, at which time she will officially take possession of her inheritance. Meanwhile, her "guardian" and his confederates constantly come up with schemes to get rid of Pauline so that he can get his hands on the money himself.', 'languages': ['English'], 'released': datetime.datetime(1914, 3, 23, 0, 0), '

In [4]:
# #group : SQL에서의 GROUP BY 와 동일한 기능

for movie in movies.aggregate([{"$group": {"_id": "$director"}}, {"$sum": 1}]) :
    print(movie)

OperationFailure: Unrecognized pipeline stage name: '$sum', full error: {'ok': 0.0, 'errmsg': "Unrecognized pipeline stage name: '$sum'", 'code': 40324, 'codeName': 'Location40324'}

In [5]:
pipeline = [{"$group": {"_id": "$directors", "count": {"$sum": 1}}}]

for movie in movies.aggregate(pipeline) :
    print(movie)

{'_id': ['Frank Miller', 'Robert Rodriguez'], 'count': 1}
{'_id': ['Sebastian del Amo'], 'count': 1}
{'_id': ['Robert Cormack', 'Clyde Geronimi', 'Jack Kinney', 'Hamilton Luske', 'Joshua Meador'], 'count': 2}
{'_id': ['Nikos Panayotopoulos'], 'count': 2}
{'_id': ['Don Sikorski'], 'count': 1}
{'_id': ['Sèbastien Lifshitz'], 'count': 3}
{'_id': ['Adolfo Aristarain'], 'count': 4}
{'_id': ['Alain Chabat'], 'count': 2}
{'_id': ['Christopher Crowe'], 'count': 1}
{'_id': ['Gyèrgy Pèlfi'], 'count': 3}
{'_id': ['Tomasz Thomson'], 'count': 2}
{'_id': ['David Van Taylor'], 'count': 1}
{'_id': ['Daniel Sackheim'], 'count': 2}
{'_id': ['Adam Brooks', 'Jeremy Gillespie', 'Matthew Kennedy', 'Steven Kostanski', 'Conor Sweeney'], 'count': 1}
{'_id': ['Puri Jagannadh'], 'count': 2}
{'_id': ['Jack Nicholson'], 'count': 1}
{'_id': ['Sang-soo Hong'], 'count': 16}
{'_id': ['Geoff Murphy'], 'count': 4}
{'_id': ['Cary Joji Fukunaga'], 'count': 1}
{'_id': ['Rod Lurie'], 'count': 6}
{'_id': ['Scott Schirmer'], 

In [8]:
# $sort : 문서의 정렬을 하고 싶을 때, 복수의 정렬을 할 수 있음. 다만 좌측부터 정렬방식이 적용됨!

pipeline = [
    {"$sort": {"title": 1}}, {"$limit": 3}
]

for movie in movies.aggregate(pipeline) :
    print(movie)

{'_id': ObjectId('573a13def29313caabdb6863'), 'plot': "Set in the golden era of Grand Prix Racing '1' tells the story of a generation of charismatic drivers who raced on the edge, risking their lives during Formula 1's deadliest period, and the men who stood up and changed the sport forever.", 'genres': ['Documentary'], 'runtime': 112, 'cast': ['Michael Fassbender', 'Niki Lauda', 'Lewis Hamilton', 'Michael Schumacher'], 'num_mflix_comments': 1, 'poster': 'https://m.media-amazon.com/images/M/MV5BMjAxNDI5NTMxMF5BMl5BanBnXkFtZTgwMDMzMjg4MTE@._V1_SY1000_SX677_AL_.jpg', 'title': 1, 'fullplot': "Set in the golden era of Grand Prix Racing '1' tells the story of a generation of charismatic drivers who raced on the edge, risking their lives during Formula 1's deadliest period, and the men who stood up and changed the sport forever.", 'languages': ['English'], 'released': datetime.datetime(2013, 10, 1, 0, 0), 'directors': ['Paul Crowder'], 'writers': ['Mark Monroe'], 'awards': {'wins': 0, 'nomin

In [ ]:
# project : 특정

pipeline = [
    {"$project": {"_id": 0, "title": 1, "genres": 1}}, {"$limit": 3}
]

for movie in movies.aggregate(pipeline) :
    print(movie)

In [10]:
# $unwined : 배열 안에 입력된 값을 추출해서 출력하고자 할 때

pipeline = [
    {"$unwind": "$genres", {"$limit": 3}}
]

for movie in movies.aggregate(pipeline) :
    print(movie)

SyntaxError: ':' expected after dictionary key (1476552500.py, line 4)

In [12]:
# $group & 집계연산자 응용 패턴

pipeline = [
    {"$group": {"_id": "$directors", "count": {"$sum": 1}}}
]

for movie in movies.aggregate(pipeline) :
    print(movie)

{'_id': ['Richard Pryor'], 'count': 1}
{'_id': ['Charles Grosvenor'], 'count': 7}
{'_id': ['Daniel Calparsoro'], 'count': 2}
{'_id': ['Andrew Fleming'], 'count': 7}
{'_id': ['Gary J. Tunnicliffe'], 'count': 1}
{'_id': ['Richard Donner'], 'count': 14}
{'_id': ['Lise Birk Pedersen'], 'count': 1}
{'_id': ['Thomas Arslan'], 'count': 1}
{'_id': ['Charlie Stratton'], 'count': 1}
{'_id': ['Laurent Tirard'], 'count': 3}
{'_id': ['Stacy Cochran'], 'count': 1}
{'_id': ['Chris Butler', 'Sam Fell'], 'count': 1}
{'_id': ['Bernard Borderie'], 'count': 1}
{'_id': ['Joseph Brooks'], 'count': 1}
{'_id': ['Darnell Martin'], 'count': 3}
{'_id': ['Lloyd Kaufman'], 'count': 2}
{'_id': ['Jacques-Yves Cousteau', 'Louis Malle'], 'count': 1}
{'_id': ['S.A. Halewood'], 'count': 1}
{'_id': ['Shawn Ku'], 'count': 2}
{'_id': ['Ole Gièver'], 'count': 1}
{'_id': ['Andrew Levitas'], 'count': 1}
{'_id': ['Frank Capra', 'Hugh Stewart', 'John Huston'], 'count': 1}
{'_id': ['Règis Wargnier'], 'count': 3}
{'_id': ['Arie P

In [13]:
# $group & 집계연산자 응용 패턴

pipeline = [
    {"$group": {"_id": "$directors", "average_rating": {"$avg": "$imdb.rating"}}},
    {"$limit": 5}
]

for movie in movies.aggregate(pipeline) :
    print(movie)

{'_id': ['Michael Curtiz', 'John Wayne'], 'average_rating': 6.9}
{'_id': ['Peter Chelsom'], 'average_rating': 6.344444444444445}
{'_id': ['Dominique Abel', 'Fiona Gordon', 'Bruno Romy'], 'average_rating': 6.566666666666666}
{'_id': ['Juris Kursietis'], 'average_rating': 6.9}
{'_id': ['Robert Stevenson'], 'average_rating': 6.840000000000001}


In [14]:
# 컬렉션에 있는 영화의 갯수를 계산하세요!
# 특정 컬렉션에 존재하는 갯수를

movie_count = movies.count_documents({})
print("영화 수 : ", movie_count)



영화 수 :  23539


In [15]:
# 평균 영화 길이를 찾으세요.
# 영화 필드를 그룹화하고,  해당 영화들의 런타임값을 활용해서 평균값을 구한다.
# $avg

movies.aggregate([{"$group": {"_id": "$runtime", "avg_length"}}])

for movie in ni==mo



SyntaxError: ':' expected after dictionary key (3732897868.py, line 5)

In [39]:
# 각 장르에 대한 영화 갯수를 계산하세요!
# 장르를 기준으로 갯수를 출력 => $group // $sum

# pipeline = [
#     {"$group": {"_id": "$genres", "count": {"$sum": 1}}}
# ]

# for movie in movies.aggregate(pipeline) :
#     print(movie)
genre_counts = list(
movies.aggregate([
    {"$unwind": "$genres"},
    {"$group": {"_id": "$genres", "count": {"$sum": 1}}}
]))

for genre in genre_counts :
    print(f"장르 : {genre["_id"]}, 영화 수 : {genre["count"]}")

장르 : Biography, 영화 수 : 1404
장르 : Short, 영화 수 : 478
장르 : Mystery, 영화 수 : 1259
장르 : War, 영화 수 : 794
장르 : Horror, 영화 수 : 1703
장르 : Film-Noir, 영화 수 : 105
장르 : Thriller, 영화 수 : 2658
장르 : Comedy, 영화 수 : 7024
장르 : Musical, 영화 수 : 487
장르 : Family, 영화 수 : 1311
장르 : Animation, 영화 수 : 971
장르 : Western, 영화 수 : 274
장르 : History, 영화 수 : 999
장르 : Crime, 영화 수 : 2678
장르 : Documentary, 영화 수 : 2129
장르 : Sport, 영화 수 : 390
장르 : Talk-Show, 영화 수 : 1
장르 : Music, 영화 수 : 840
장르 : News, 영화 수 : 51
장르 : Fantasy, 영화 수 : 1153
장르 : Action, 영화 수 : 2539
장르 : Romance, 영화 수 : 3665
장르 : Drama, 영화 수 : 13789
장르 : Adventure, 영화 수 : 2045
장르 : Sci-Fi, 영화 수 : 1034


In [34]:
# 2015년 이후에 개봉한 영화를 제목으로 정렬하여 나열해주세요.

for movie in movies.find(
    {"year": {"$lt": 2015}},
    {"_id": 0, "title": 1, "year": 1}
) :
    print(movie)

{'title': 'Blacksmith Scene', 'year': 1893}
{'title': 'The Great Train Robbery', 'year': 1903}
{'title': 'The Land Beyond the Sunset', 'year': 1912}
{'title': 'A Corner in Wheat', 'year': 1909}
{'title': 'Winsor McCay, the Famous Cartoonist of the N.Y. Herald and His Moving Comics', 'year': 1911}
{'title': 'Traffic in Souls', 'year': 1913}
{'title': 'Gertie the Dinosaur', 'year': 1914}
{'title': 'In the Land of the Head Hunters', 'year': 1914}
{'title': 'The Perils of Pauline', 'year': 1914}
{'title': 'The Birth of a Nation', 'year': 1915}
{'title': 'The Cheat', 'year': 1915}
{'title': 'The Italian', 'year': 1915}
{'title': 'Regeneration', 'year': 1915}
{'title': 'Les vampires', 'year': 1915}
{'title': 'Civilization', 'year': 1916}
{'title': "Hell's Hinges", 'year': 1916}
{'title': "Intolerance: Love's Struggle Throughout the Ages", 'year': 1916}
{'title': 'Where Are My Children?', 'year': 1916}
{'title': 'The Immigrant', 'year': 1917}
{'title': 'The Poor Little Rich Girl', 'year': 191

In [35]:
# aggregation framework()

pipeline = [
    {"$match": {"year": {"$gt": 2015}}},
    {"$sort": {"title": 1}},
    {"$project": {"_id": 0, "title": 1}}
]

for movie in movies.aggregation(pipeline) :
    print(f"영화제목 : {movie["title"]}")

TypeError: 'Collection' object is not callable. If you meant to call the 'aggregation' method on a 'Collection' object it is failing because no such method exists.

In [38]:
# 가장 많은 영화를 제작하는 상위 5개 국가를 찾아주세요.

pipeline = [
    {"$unwind": "$countries"},
    {"$group": {"_id": "$countries", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 5}
]

for country in movies.aggregation(pipeline) :
    print(country)

TypeError: 'Collection' object is not callable. If you meant to call the 'aggregation' method on a 'Collection' object it is failing because no such method exists.

In [40]:
# 2000년 이후 영화의 연도별 평균 IMDB 평점을 찾으세요.

pipeline = [
    {"$match": {"year": {"$gt": 2000}}},
    {"$group": {"_id": "$year", "avg": {"$avg": 1}}},
    {"$sort": {"avg": -1}},
    {"$limit": 5}
]

movies.aggregation(pipeline)
# for year 

TypeError: 'Collection' object is not callable. If you meant to call the 'aggregation' method on a 'Collection' object it is failing because no such method exists.